=================================================================================================

In [1]:
#Tworzenie tabeli w PosgreSQL
import psycopg2

conn = psycopg2.connect(
    host="postgresql_streaming_lab",
    port=5432,
    dbname="mydb",
    user="myuser",
    password="myuserpass"
)
cursor = conn.cursor()

# ── Tworzenie rozszerzenia i tabeli ──────────────────────────────────────────
cursor.execute("CREATE EXTENSION IF NOT EXISTS vector;")

cursor.execute("""
    CREATE TABLE IF NOT EXISTS transactions (
        id               SERIAL          PRIMARY KEY,
        sender           VARCHAR(100)    NOT NULL,
        receiver         VARCHAR(100)    NOT NULL,
        amount           NUMERIC(10,2)   NOT NULL,
        timestamp        TIMESTAMP       NOT NULL,
        device_sender    VARCHAR(100),
        device_receiver  VARCHAR(100),
        title            TEXT,
        title_embedding  vector(384)
    );
""")

conn.commit()
print("Rozszerzenie vector i tabela transactions utworzone.")

cursor.close()
conn.close()

Rozszerzenie vector i tabela transactions utworzone.


In [2]:
#Consumer A
from kafka import KafkaConsumer
from sentence_transformers import SentenceTransformer
import json

model = SentenceTransformer('all-MiniLM-L6-v2')

topic = 'transactions'

consumer = KafkaConsumer(
    topic,
    bootstrap_servers='kafka_streaming_lab:9092',
    auto_offset_reset='earliest',
    enable_auto_commit=True,
    group_id='customA-group'
)

# PostgreSQL
conn = psycopg2.connect(
    dbname='mydb',
    user='myuser',
    password='myuserpass',
    host='postgres',
    port=5432
)

cursor = conn.cursor()

print("Kafka-to-Postgres processor running...")

try:
    for msg in consumer:

        # decode JSON
        data = json.loads(msg.value.decode('utf-8'))

        # jeśli pojedynczy rekord
        if isinstance(data, dict):
            data = [data]

        for t in data:

            # embedding
            embedding = model.encode(t['title'])

            # insert
            cursor.execute(
                """
                INSERT INTO transactions (
                    sender,
                    receiver,
                    amount,
                    timestamp,
                    device_sender,
                    device_receiver,
                    title,
                    title_embedding
                )
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
                """,
                (
                    t['sender'],
                    t['receiver'],
                    t['amount'],
                    t['timestamp'],
                    t['device_sender'],
                    t['device_receiver'],
                    t['title'],
                    embedding.tolist()
                )
            )

            conn.commit()

            print(f"Inserted into DB: {t}")

except KeyboardInterrupt:
    print("Stopping consumer...")

finally:
    consumer.close()
    cursor.close()
    conn.close()

    print("Connections closed.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Kafka-to-Postgres processor running...
Inserted into DB: {'sender': 'user1', 'receiver': 'user2', 'amount': 120.5, 'timestamp': '2026-05-10T10:00:00', 'device_sender': 'devA', 'device_receiver': 'devB', 'title': 'zwrot za obiad'}
Inserted into DB: {'sender': 'user3', 'receiver': 'user4', 'amount': 75.0, 'timestamp': '2026-05-09T11:00:00', 'device_sender': 'devC', 'device_receiver': 'devD', 'title': 'oddaje pieniadze'}
Inserted into DB: {'sender': 'user5', 'receiver': 'user6', 'amount': 200.0, 'timestamp': '2026-05-08T09:30:00', 'device_sender': 'devE', 'device_receiver': 'devF', 'title': 'zwrot kosztow'}
Inserted into DB: {'sender': 'user2', 'receiver': 'user1', 'amount': 50.0, 'timestamp': '2026-05-07T14:00:00', 'device_sender': 'devB', 'device_receiver': 'devA', 'title': 'zwrot za paliwo'}
Inserted into DB: {'sender': 'user7', 'receiver': 'user8', 'amount': 300.0, 'timestamp': '2026-05-06T16:00:00', 'device_sender': 'devG', 'device_receiver': 'devH', 'title': 'oddaje dlug'}
Inserted 

In [3]:
#Zapytania sprawdzające: Znajdź przelewy zawierające kontekst: jedzenia oraz zwrotu środków w zakresie 30 - 150

# Generuj wektory zapytań 
vec_jedzenie = "[" + ",".join(map(str, model.encode("jedzenie restauracja obiad pizza").tolist())) + "]"
vec_zwrot    = "[" + ",".join(map(str, model.encode("zwrot środków zwrot oddaje pieniądze refund").tolist())) + "]"

# Połączenie z PostgreSQL 
conn = psycopg2.connect(
    host="postgresql_streaming_lab",
    port=5432,
    dbname="mydb",
    user="myuser",
    password="myuserpass"
)
cursor = conn.cursor()

#Zapytanie pgvector
query = """
    SELECT
        id,
        sender,
        receiver,
        amount,
        timestamp,
        title,
        ROUND((1 - (title_embedding <=> %s::vector))::numeric, 3) AS sim_jedzenie,
        ROUND((1 - (title_embedding <=> %s::vector))::numeric, 3) AS sim_zwrot,
        CASE
            WHEN (1 - (title_embedding <=> %s::vector)) >= 0.45 THEN 'jedzenie'
            ELSE 'zwrot 30-150'
        END AS match_type
    FROM transactions
    WHERE
        -- kontekst jedzenia
        (1 - (title_embedding <=> %s::vector)) >= 0.45
        OR
        -- zwrot środków w zakresie 30-150
        (
            (1 - (title_embedding <=> %s::vector)) >= 0.45
            AND amount BETWEEN 30 AND 150
        )
    ORDER BY
        GREATEST(
            (1 - (title_embedding <=> %s::vector)),
            (1 - (title_embedding <=> %s::vector))
        ) DESC;
"""

cursor.execute(query, (
    vec_jedzenie,   # sim_jedzenie (SELECT)
    vec_zwrot,      # sim_zwrot    (SELECT)
    vec_jedzenie,   # match_type   (CASE)
    vec_jedzenie,   # WHERE jedzenie
    vec_zwrot,      # WHERE zwrot
    vec_jedzenie,   # ORDER BY GREATEST
    vec_zwrot,      # ORDER BY GREATEST
))

rows = cursor.fetchall()

# Wyniki
print(f"Znaleziono: {len(rows)} rekordów\n")
print(f"{'ID':<6} {'sender':<10} {'receiver':<10} {'amount':>8} {'title':<25} {'sim_food':>9} {'sim_zwrot':>9} {'match':<12}")
print("-" * 95)

for row in rows:
    id_, sender, receiver, amount, timestamp, title, sim_food, sim_zwrot, match = row
    print(f"{id_:<6} {sender:<10} {receiver:<10} {float(amount):>8.2f} {title:<25} {float(sim_food):>9.3f} {float(sim_zwrot):>9.3f} {match:<12}")

cursor.close()
conn.close()

Znaleziono: 7 rekordów

ID     sender     receiver     amount title                      sim_food sim_zwrot match       
-----------------------------------------------------------------------------------------------
19     user9      user11       125.00 jedzenie                      0.639     0.268 jedzenie    
28     user18     user20       215.00 jedzenie                      0.639     0.268 jedzenie    
24     user14     user16       175.00 jedzenie                      0.639     0.268 jedzenie    
11     user1      user3         45.00 pizza                         0.549     0.186 jedzenie    
2      user3      user4         75.00 oddaje pieniadze              0.491     0.520 jedzenie    
1      user1      user2        120.50 zwrot za obiad                0.437     0.501 zwrot 30-150
4      user2      user1         50.00 zwrot za paliwo               0.352     0.480 zwrot 30-150
